# Musanga — Block 13A.2: independent dense-teacher audit
This is a **standalone companion notebook**. It opens the ten saved 13A diagnostic windows and compares their native 4.77 m UAV probabilities to the verified 10 m cached references. It does **not** rerun UAV inference, raster warping, or CNN training.

Run the single code cell. If 13A saved fewer than ten files, finish 13A first.

In [ ]:
# 13A.2 — Check existing diagnostic patches without reopening huge UAV files.
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from pathlib import Path
from IPython.display import display
from pathlib import Path
roots=[Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
       Path('/content/drive/MyDrive/Musanga/Final')]
FINAL=next((x for x in roots if x.is_dir()),None)
if FINAL is None:
    from google.colab import drive
    drive.mount('/content/drive')
    FINAL=next((x for x in roots if x.is_dir()),None)
assert FINAL is not None,'Cannot find Musanga Final folder in Google Drive'
AUDIT13=FINAL/'Outputs/V3_1/06_spatial_planet_embeddings/13_dense_planet_unet/13A_registration_audit'
assert AUDIT13.is_dir(),f'Run 13A in the main NB06 notebook first: {AUDIT13}'
print('Reading cached 13A samples:',AUDIT13,flush=True)
audit_files13=sorted(AUDIT13.glob('*.npz'))
assert len(audit_files13)==10,('Expected ten cached 13A mosaic pilots',len(audit_files13))
def local_smooth13(x,valid,sigma=1.0):
    v=valid.astype('float64')
    num=gaussian_filter(np.where(valid,x,0.0).astype('float64'),sigma=sigma)
    den=gaussian_filter(v,sigma=sigma)
    return np.divide(num,den,out=np.full_like(num,np.nan),where=den>.6),den
records13v=[]
for f in audit_files13:
    with np.load(f) as z:
        meta=__import__('json').loads(str(z['metadata'].item()))
        predicted=z['decoded_teacher'].astype('float64')
        reference=z['cached10'].astype('float64')
        valid=(z['valid'].astype(bool)&np.isfinite(predicted)&np.isfinite(reference)&
               (predicted>=0)&(predicted<=1)&(reference>=0)&(reference<=1))
        assert meta['selected_encoding']=='percent_10000',(f,meta['selected_encoding'])
        assert valid.sum()>200,(f,valid.sum())
        difference=predicted[valid]-reference[valid]
        p_s,wt=local_smooth13(predicted,valid,sigma=1.15)
        c_s,_=local_smooth13(reference,valid,sigma=1.15)
        core=valid&(wt>.95)&np.isfinite(p_s)&np.isfinite(c_s)
        assert core.sum()>100
        smdiff=p_s[core]-c_s[core]
        corr=np.corrcoef(p_s[core],c_s[core])[0,1] if (
            np.std(p_s[core])>1e-7 and np.std(c_s[core])>1e-7) else np.nan
        records13v.append(dict(
            site=meta['site'],mosaic=meta['mosaic'],pilot=meta['pilot'],
            n_valid=int(valid.sum()),encoding=meta['selected_encoding'],
            native_vs_10m_MAE_pp=100*np.mean(np.abs(difference)),
            signed_bias_pp=100*np.mean(difference),
            native_mean_pct=100*np.mean(predicted[valid]),
            cached_10m_mean_pct=100*np.mean(reference[valid]),
            smoothed_MAE_pp=100*np.mean(np.abs(smdiff)),
            smoothed_corr=corr,
            within_mean_2pp=bool(abs(np.mean(difference))<=.02)
        ))
report13v=pd.DataFrame(records13v).sort_values(['site','mosaic','pilot'])
report13v.to_csv(AUDIT13/'13A2_scale_agreement.csv',index=False)
display(report13v.round(3))
print('All pilots have consistent 0-10000 encoding:',(report13v.encoding=='percent_10000').all())
print('Mean bias within 2 percentage points:',
      int(report13v.within_mean_2pp.sum()),'/',len(report13v))
print('WARNING: similarity to a 10m reference is not proof of subpixel registration.')
print('13A2 files saved in',AUDIT13)
